## Scraping Amazon Website to extrect data of sneakers

## Importing Required Libraries
This step imports necessary libraries:
- `requests` → to send HTTP requests to Amazon
- `BeautifulSoup` → to parse HTML content
- `numpy` → for numerical operations
- `pandas` → for data handling and DataFrame creation

In [1]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import time
import random
import re

## Initializing Data Storage Lists
Empty lists are created to store extracted data such as:
- ASIN (unique product ID)
- Title (product name)
- Prices (discounted and original)
- Ratings and review counts

In [2]:


asin_data = []     # Stores unique product ID
title_data = []   # Stores product titles
price_data = []   # Stores selling prices
original_price_data = []  # Stores MRP (required for calculating discount %)
ratings_data = []   # Stores ratings of product
review_data = []   # Stores number of reviews


session = requests.Session()                                       # Creates perdidtent session,makes performance faster,less likely to get blocked
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/123.0.0.0 Safari/537.36",
    "Accept-Language": "en-IN,en;q=0.9"
}                                                                  # tells server that requests is coming from a real browser , avoids bot detection

for page in range(1, 21):
    print(f"Scraping page {page}")                                 #helps track progress and detecting failures
    url = f"https://www.amazon.in/s?k=sneakers&page={page}"        #dynamically builds url for each page
    print(url)
    time.sleep(random.uniform(3, 6))                               #random delay(human-like behaviour),reduces chances of getting blocked
    response = session.get(url, headers=headers, timeout=15)       # Sends HTTP requests, timeout prevents hanging if requests fails
    html = response.text                                           # Extract raw HTML, needed for parsing

    if "captcha" in html.lower():                                  #Detects if u get blocked,because Amazon often returns captcha when found suspicious
        print("Blocked by Amazon")
        break

    soup = BeautifulSoup(html, "lxml")                             # parses HTML,lxml parser is more faster and robust than default
    containers = soup.find_all("div", {"data-component-type": "s-search-result"})
    # Each product is inside this 'div'
    
    for c in containers:                                          #Iterates over each product block,extracts data per product
        
        #  ASIN 
        
        asin = c.get("data-asin")
        if not asin:
            continue  # Skips ads/empty containers
        asin_data.append(asin)

        #  TITLE
        
        title = c.find("h2")  # Title is inside <h2> tag(Std Amazon Structure)
        title_data.append(title.get_text(strip=True) if title else np.nan)
        # Extract text , remove spaces if missing then stores NaN value for consistency
        
        # PRICE (Discounted) 
        
        price = c.find("span", class_="a-price-whole")
        price_data.append(price.get_text(strip=True) if price else np.nan)
        # clean extraction , ensures no crashes if price missing
        
        #  ORIGINAL PRICE (MRP)
        
        original = c.select_one("span.a-price.a-text-price span.a-offscreen")
        # This is CSS selector,specifically targets MRP
        if original:
    
            original_price_data.append(original.get_text(strip=True).replace("₹", "").replace(",", "").strip())  # removes currency symbol and commas for numeric conversion
        else:
            original_price_data.append(np.nan)
            # missing MRP(store NaN for analysis consistency
            
        # RATING
        
        rating = c.find("span", class_="a-icon-alt")
        ratings_data.append(rating.get_text(strip=True) if rating else np.nan)
        #Storing raw Rating text, will clean later
       
        # REVIEW COUNT 
       
        review_count = np.nan

        # Amazon wraps review count in an <a> tag with aria-label like "1,234 ratings"
        review_tag = c.select_one("a[href*='customerReviews'], span[aria-label*='stars'] ~ span a")             #  Targets anchor tag containing review count
        if review_tag:
            text = review_tag.get_text(strip=True)                                                              #  Extract text like "1,234"
            m = re.search(r"[\d,]+", text)
            if m:
                review_count = m.group()

        
        if pd.isna(review_count):                         
            for tag in c.find_all("span", attrs={"aria-label": True}):
                label = tag.get("aria-label", "")
                # Skip rating stars label ("4.2 out of 5 stars"), grab review count label
                if "out of" not in label.lower() and re.search(r"[\d,]+", label):
                    m = re.search(r"[\d,]+", label)
                    if m:
                        review_count = m.group()
                        break

       
        if pd.isna(review_count):
            for span in c.find_all("span", class_="a-size-base"):
                text = span.get_text(strip=True)
                if re.fullmatch(r"[\d,]+", text):          # must be ONLY digits/commas
                    review_count = text
                    break

        review_data.append(review_count)




Scraping page 1
https://www.amazon.in/s?k=sneakers&page=1
Scraping page 2
https://www.amazon.in/s?k=sneakers&page=2
Scraping page 3
https://www.amazon.in/s?k=sneakers&page=3
Scraping page 4
https://www.amazon.in/s?k=sneakers&page=4
Scraping page 5
https://www.amazon.in/s?k=sneakers&page=5
Scraping page 6
https://www.amazon.in/s?k=sneakers&page=6
Scraping page 7
https://www.amazon.in/s?k=sneakers&page=7
Scraping page 8
https://www.amazon.in/s?k=sneakers&page=8
Scraping page 9
https://www.amazon.in/s?k=sneakers&page=9
Scraping page 10
https://www.amazon.in/s?k=sneakers&page=10
Scraping page 11
https://www.amazon.in/s?k=sneakers&page=11
Scraping page 12
https://www.amazon.in/s?k=sneakers&page=12
Scraping page 13
https://www.amazon.in/s?k=sneakers&page=13
Scraping page 14
https://www.amazon.in/s?k=sneakers&page=14
Scraping page 15
https://www.amazon.in/s?k=sneakers&page=15
Scraping page 16
https://www.amazon.in/s?k=sneakers&page=16
Scraping page 17
https://www.amazon.in/s?k=sneakers&page=1

## Web Scraping Process
Data is extracted from Amazon using:
- HTML tags and attributes identified via inspection
- Looping through product containers
- Extracting relevant fields like title, price, rating, and ASIN

Each value is appended to its respective list.

In [18]:
df = pd.DataFrame({
    "ASIN":           asin_data,
    "Title":          title_data,
    "Discounted Price": price_data,
    "Original Price": original_price_data,
    "Rating":         ratings_data,
    "Review Count":   review_data
})


# NEW COLUMN: Discount %

df["Discounted Price"] = pd.to_numeric(df["Discounted Price"].str.replace(",", ""), errors="coerce")
df["Original Price"]   = pd.to_numeric(df["Original Price"].str.replace(",", ""),   errors="coerce")

df["Discount %"] = np.where(
    df["Original Price"].notna() & df["Discounted Price"].notna() & (df["Original Price"] > 0),
    ((df["Original Price"] - df["Discounted Price"]) / df["Original Price"] * 100).round(1),
    np.nan
)

# NEW Rating Category

df['Rating'] = df['Rating'].str.extract(r'(\d+\.\d+)').astype(float)

df['Rating Category'] = pd.cut(
    df['Rating'],
    bins=[0, 3.5, 4.0, 4.5, 5],
    labels=['Poor', 'Average', 'Good', 'Excellent']
)

# Popularity Category

df['Review Count'] = df['Review Count'].str.replace(',', '')
df['Review Count'] = pd.to_numeric(df['Review Count'], errors='coerce')

df['Popularity'] = pd.cut(
    df['Review Count'],
    bins=[0, 100, 500, 2000, 5000],
    labels=['Low', 'Moderate', 'High', 'Very High']
)

print(df.head())
print(df.shape)

         ASIN    Title  Discounted Price  Original Price  Rating  \
0  B098PC5X7Y     Nike            4995.0             NaN     3.8   
1  B0FDVZX9LF     Nike            6495.0             NaN     3.7   
2  B0FYQMG3VM  Boldfit            1499.0          4499.0     4.4   
3  B0GFWRXCP2  Boldfit            1999.0          3999.0     4.4   
4  B0DJD6NGT3    ASIAN             674.0          1099.0     3.7   

   Review Count  Discount % Rating Category Popularity  
0           4.0         NaN         Average        Low  
1          15.0         NaN         Average        Low  
2         156.0        66.7            Good   Moderate  
3          45.0        50.0            Good        Low  
4         890.0        38.7         Average       High  
(546, 9)


## Preview of Dataset
Displays the scraped dataset to verify if data has been collected correctly.

In [19]:
df

,ASIN,Title,Discounted Price,Original Price,Rating,Review Count,Discount %,Rating Category,Popularity
0,B098PC5X7Y,Nike,4995.0,NaN,3.8,4.0,NaN,Average,Low
1,B0FDVZX9LF,Nike,6495.0,NaN,3.7,15.0,NaN,Average,Low
2,B0FYQMG3VM,Boldfit,1499.0,4499.0,4.4,156.0,66.7,Good,Moderate
3,B0GFWRXCP2,Boldfit,1999.0,3999.0,4.4,45.0,50.0,Good,Low
4,B0DJD6NGT3,ASIAN,674.0,1099.0,3.7,890.0,38.7,Average,High
...,...,...,...,...,...,...,...,...,...
541,B0F5X55PVH,Lotto,4749.0,4999.0,4.4,25.0,5.0,Good,Low
542,B0FG2WCN1F,PUMA,3299.0,5999.0,4.0,56.0,45.0,Average,Low
543,B0FDQPTMD4,Liberty,889.0,2499.0,3.7,76.0,64.4,Average,Low
544,B0GPQN62CK,Lotto,5499.0,NaN,NaN,NaN,NaN,NaN,NaN


## Checking Missing Values
This step identifies missing (NaN) values in each column to understand data quality issues.

In [20]:
df.isna().sum()

ASIN                  0
Title                 0
Discounted Price      2
Original Price      138
Rating               63
Review Count         44
Discount %          138
Rating Category      63
Popularity           44
dtype: int64

# Treating NaN values

## Handling Missing Values
To ensure clean analysis, rows with missing values are removed from the dataset.

In [24]:
df_new = df.dropna(axis=0)

## Cleaned Dataset
A new dataset is created after removing all rows containing missing values.

In [25]:
df_new.isna().sum()

ASIN                0
Title               0
Discounted Price    0
Original Price      0
Rating              0
Review Count        0
Discount %          0
Rating Category     0
Popularity          0
dtype: int64

In [26]:
df_new

,ASIN,Title,Discounted Price,Original Price,Rating,Review Count,Discount %,Rating Category,Popularity
2,B0FYQMG3VM,Boldfit,1499.0,4499.0,4.4,156.0,66.7,Good,Moderate
3,B0GFWRXCP2,Boldfit,1999.0,3999.0,4.4,45.0,50.0,Good,Low
4,B0DJD6NGT3,ASIAN,674.0,1099.0,3.7,890.0,38.7,Average,High
5,B07Y548LQW,SPARX,662.0,849.0,4.2,17.0,22.0,Good,Low
6,B0GFX69MTD,Campus,999.0,2499.0,4.7,52.0,60.0,Excellent,Low
...,...,...,...,...,...,...,...,...,...
540,B0BJ2PLS8C,Red Chief,1599.0,4099.0,3.2,12.0,61.0,Poor,Low
541,B0F5X55PVH,Lotto,4749.0,4999.0,4.4,25.0,5.0,Good,Low
542,B0FG2WCN1F,PUMA,3299.0,5999.0,4.0,56.0,45.0,Average,Low
543,B0FDQPTMD4,Liberty,889.0,2499.0,3.7,76.0,64.4,Average,Low


In [27]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392 entries, 2 to 545
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   ASIN              392 non-null    object  
 1   Title             392 non-null    object  
 2   Discounted Price  392 non-null    float64 
 3   Original Price    392 non-null    float64 
 4   Rating            392 non-null    float64 
 5   Review Count      392 non-null    float64 
 6   Discount %        392 non-null    float64 
 7   Rating Category   392 non-null    category
 8   Popularity        392 non-null    category
dtypes: category(2), float64(5), object(2)
memory usage: 25.7+ KB


## Dataset Information
Provides summary of:
- Data types
- Non-null counts
- Column structure

In [16]:
df_new['Rating'] = df_new['Rating'].astype("float64")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20464\720481756.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['Rating'] = df_new['Rating'].astype("float64")


## Data Type Conversion
The 'Rating' column is converted into numeric format for accurate analysis and visualization.

In [28]:
df_new.dtypes

ASIN                  object
Title                 object
Discounted Price     float64
Original Price       float64
Rating               float64
Review Count         float64
Discount %           float64
Rating Category     category
Popularity          category
dtype: object

# Converting to CSV file

In [30]:
df_new.to_csv('Sneakers_data.csv',index = False)